# FS09-10 hardened


In [ ]:
import os, json, math, random, time, re
from pathlib import Path
import numpy as np
os.environ.pop('CUDA_VISIBLE_DEVICES', None)
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
OUT=Path('/kaggle/working'); FIG=OUT/'figures'; RES=OUT/'results'
FIG.mkdir(parents=True, exist_ok=True); RES.mkdir(parents=True, exist_ok=True)
device=torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print('device',device,'gpus',torch.cuda.device_count() if torch.cuda.is_available() else 0)
PROGRESS={}; GATES={}
def gate(name, ok, detail=''):
    GATES[name]=bool(ok); print(('PASS' if ok else 'FAIL'), name, detail)
    if not ok: raise AssertionError(f'ACCEPTANCE FAILED: {name} {detail}')
def make_shape_image(kind, size=32):
    img=np.ones((size,size,3),np.float32)*0.95
    yy,xx=np.mgrid[0:size,0:size]; cy,cx=size//2,size//2
    if kind=='red_circle':
        m=(yy-cy)**2+(xx-cx)**2<=(size*0.28)**2; img[m]=(0.9,0.15,0.12)
    elif kind=='blue_square':
        m=(np.abs(yy-cy)<size*0.25)&(np.abs(xx-cx)<size*0.25); img[m]=(0.15,0.25,0.85)
    elif kind=='green_triangle':
        top=cy-int(size*0.28); bot=cy+int(size*0.30)
        for y in range(max(0,top),min(size,bot)):
            half=int((y-top)/(bot-top+1e-6)*size*0.30)
            img[y, max(0,cx-half):min(size,cx+half+1)]=(0.15,0.75,0.25)
    else: raise ValueError(kind)
    return img
CLASSES=['red_circle','blue_square','green_triangle']
c2i={c:i for i,c in enumerate(CLASSES)}


## FS09


In [ ]:
# Video classification: make classes VERY separable + train/eval same distribution
VIDEO_KINDS=['circle_right','square_down','triangle_pulse']
LABELS={
 'circle_right':'circle moving right',
 'square_down':'square moving down',
 'triangle_pulse':'triangle color pulse',
}

def make_video(kind, T=8, size=32):
    frames=[]
    for t in range(T):
        img=np.ones((size,size,3),np.float32)*0.95
        yy,xx=np.mgrid[0:size,0:size]
        if kind=='circle_right':
            cx=int(size*(0.2+0.6*t/(T-1))); cy=size//2; r=size*0.16
            m=(yy-cy)**2+(xx-cx)**2<=r**2; img[m]=(0.9,0.2,0.15)
        elif kind=='square_down':
            cy=int(size*(0.2+0.6*t/(T-1))); cx=size//2; s=int(size*0.16)
            m=(np.abs(yy-cy)<s)&(np.abs(xx-cx)<s); img[m]=(0.15,0.25,0.85)
        else:
            cy,cx=size//2,size//2
            sc=0.14+0.08*(t/(T-1))
            top=int(cy-size*sc); bot=int(cy+size*sc*1.2)
            for y in range(max(0,top),min(size,bot)):
                half=int((y-top)/(bot-top+1e-6)*size*sc)
                col=(0.15,0.75,0.25) if t%2==0 else (0.9,0.75,0.1)
                img[y, max(0,cx-half):min(size,cx+half+1)]=col
        frames.append(img)
    return np.stack(frames)  # T,H,W,3

class VideoEnc(nn.Module):
    def __init__(self,n=3):
        super().__init__()
        self.frame=nn.Sequential(nn.Conv2d(3,32,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1),nn.ReLU(),nn.AdaptiveAvgPool2d(1),nn.Flatten())
        self.temporal=nn.GRU(64,64,batch_first=True)
        self.head=nn.Linear(64,n)
    def forward(self,x):  # B,T,3,H,W
        B,T,C,H,W=x.shape
        f=self.frame(x.reshape(B*T,C,H,W)).reshape(B,T,-1)
        o,_=self.temporal(f)
        return self.head(o.mean(1))  # mean pool more stable than last only

def vid_tensor(kind, noise=0.0):
    fr=make_video(kind)
    if noise>0: fr=np.clip(fr+noise*np.random.randn(*fr.shape).astype(np.float32),0,1)
    return torch.tensor(fr.transpose(0,3,1,2),dtype=torch.float32)  # T,3,H,W

xs,ys=[],[]
for k in VIDEO_KINDS:
    for _ in range(200):
        xs.append(vid_tensor(k, noise=0.02)); ys.append(VIDEO_KINDS.index(k))
X=torch.stack(xs); Y=torch.tensor(ys)
perm=torch.randperm(len(X)); ntr=int(0.8*len(X))
Xtr,Ytr,Xva,Yva=X[perm[:ntr]],Y[perm[:ntr]],X[perm[ntr:]],Y[perm[ntr:]]
venc=VideoEnc().to(device); opt=torch.optim.Adam(venc.parameters(),lr=1e-3)
hist9=[]
for epoch in range(1,30):
    venc.train(); losses=[]; accs=[]
    perm=torch.randperm(len(Xtr))
    for i in range(0,len(Xtr),32):
        b=perm[i:i+32]; xb,yb=Xtr[b].to(device),Ytr[b].to(device)
        opt.zero_grad(set_to_none=True)
        lg=venc(xb); loss=F.cross_entropy(lg,yb); loss.backward(); opt.step()
        losses.append(loss.item()); accs.append((lg.argmax(1)==yb).float().mean().item())
    venc.eval()
    with torch.no_grad():
        vacc=(venc(Xva.to(device)).argmax(1)==Yva.to(device)).float().mean().item()
    hist9.append({'epoch':epoch,'loss':round(float(np.mean(losses)),4),'val_acc':round(vacc,4)})
    if epoch%5==0: print(hist9[-1])

# clean eval WITHOUT distribution shift: noise=0.02 same as train OR noise=0
rows9=[]
venc.eval()
with torch.no_grad():
    for k in VIDEO_KINDS:
        # average over 5 clean samples
        preds=[]
        for _ in range(5):
            v=vid_tensor(k, noise=0.0).unsqueeze(0).to(device)
            preds.append(VIDEO_KINDS[venc(v).argmax(1).item()])
        # majority
        pred=max(set(preds), key=preds.count)
        rows9.append({'input':k,'pred_label':pred,'gt':LABELS[k],'pred_caption':LABELS[pred],'ok':pred==k})
clean_acc=sum(r['ok'] for r in rows9)/len(rows9)
print(rows9, clean_acc, 'val', hist9[-1]['val_acc'])
gate('FS09_val', hist9[-1]['val_acc']>=0.95, hist9[-1])
gate('FS09_clean', clean_acc>=0.999, rows9)
fig,axes=plt.subplots(3,8,figsize=(12,4.5))
for i,k in enumerate(VIDEO_KINDS):
    fr=make_video(k)
    for t in range(8):
        axes[i,t].imshow(fr[t]); axes[i,t].axis('off')
fig.tight_layout(); fig.savefig(FIG/'fs09_video.png',dpi=120); plt.close()
(RES/'fs09.json').write_text(json.dumps({'stage':'FS09','method':'frameCNN+GRU','history':hist9,'samples':rows9,'clean_acc':clean_acc,'vs_prev':'image->video motion'},indent=2))
PROGRESS['FS09']='ok'


## FS10


In [ ]:
# Highly separable tones + longer train
WORD_FREQ={'red':220.0,'blue':440.0,'green':660.0,'stop':880.0}
WORDS=list(WORD_FREQ.keys())
def synth_wav(word, sr=8000, dur=0.5, noise=0.02):
    t=np.linspace(0,dur,int(sr*dur),endpoint=False)
    f=WORD_FREQ[word]
    env=np.sin(np.pi*np.clip(t/dur,0,1))**2
    x=0.7*env*np.sin(2*np.pi*f*t)
    x=x+noise*np.random.randn(len(x))
    return x.astype(np.float32)
def logmel(x, sr=8000, n_fft=256, hop=128, n_mels=32):
    w=np.hanning(n_fft); frames=[]
    for i in range(0, len(x)-n_fft, hop):
        frame=x[i:i+n_fft]*w; frames.append(np.abs(np.fft.rfft(frame))**2)
    S=np.stack(frames,0); F=S.shape[1]
    mel=np.zeros((S.shape[0],n_mels),np.float32)
    edges=np.linspace(0,F,n_mels+1).astype(int)
    for m in range(n_mels):
        if edges[m+1]>edges[m]: mel[:,m]=S[:,edges[m]:edges[m+1]].mean(1)
    return np.log(mel+1e-6).T  # M,T

class ASRNet(nn.Module):
    def __init__(self,n=4):
        super().__init__()
        self.cnn=nn.Sequential(nn.Conv2d(1,32,3,padding=1),nn.ReLU(),nn.Conv2d(32,64,3,padding=1),nn.ReLU(),
                               nn.AdaptiveAvgPool2d((1,None)))
        self.gru=nn.GRU(64,64,batch_first=True,bidirectional=True)
        self.fc=nn.Linear(128,n)
    def forward(self,x):
        h=self.cnn(x).squeeze(2).transpose(1,2)
        o,_=self.gru(h)
        return self.fc(o.mean(1))

def mel_fixed(word, noise=0.02, T=24):
    mel=logmel(synth_wav(word, noise=noise))
    if mel.shape[1]<T: mel=np.pad(mel,((0,0),(0,T-mel.shape[1])))
    else: mel=mel[:,:T]
    return mel

xs,ys=[],[]
for w in WORDS:
    for _ in range(150):
        xs.append(mel_fixed(w)[None]); ys.append(WORDS.index(w))
X=torch.tensor(np.stack(xs),dtype=torch.float32); Y=torch.tensor(ys)
perm=torch.randperm(len(X)); ntr=int(0.8*len(X))
Xtr,Ytr,Xva,Yva=X[perm[:ntr]],Y[perm[:ntr]],X[perm[ntr:]],Y[perm[ntr:]]
asr=ASRNet().to(device); opt=torch.optim.Adam(asr.parameters(),lr=1e-3)
hist10=[]
for epoch in range(1,40):
    asr.train(); losses=[]
    perm=torch.randperm(len(Xtr))
    for i in range(0,len(Xtr),64):
        b=perm[i:i+64]; xb,yb=Xtr[b].to(device),Ytr[b].to(device)
        opt.zero_grad(set_to_none=True)
        lg=asr(xb); loss=F.cross_entropy(lg,yb); loss.backward(); opt.step(); losses.append(loss.item())
    asr.eval()
    with torch.no_grad():
        vacc=(asr(Xva.to(device)).argmax(1)==Yva.to(device)).float().mean().item()
    hist10.append({'epoch':epoch,'loss':round(float(np.mean(losses)),4),'val_acc':round(vacc,4)})
    if epoch%10==0: print(hist10[-1])

rows10=[]
asr.eval()
with torch.no_grad():
    for w in WORDS:
        # majority over 5 draws noise=0.02
        preds=[]
        for _ in range(5):
            mel=mel_fixed(w, noise=0.02)
            x=torch.tensor(mel[None,None],dtype=torch.float32,device=device)
            preds.append(WORDS[asr(x).argmax(1).item()])
        pred=max(set(preds), key=preds.count)
        rows10.append({'gt':w,'pred':pred,'ok':pred==w})
acc10=sum(r['ok'] for r in rows10)/len(rows10)
print(rows10, acc10)
gate('FS10_val', hist10[-1]['val_acc']>=0.9, hist10[-1])
gate('FS10_clean', acc10>=0.999, rows10)
fig,axes=plt.subplots(2,4,figsize=(10,4))
for i,w in enumerate(WORDS):
    wav=synth_wav(w); mel=logmel(wav)
    axes[0,i].plot(wav[:2000],lw=0.5); axes[0,i].set_title(w,fontsize=8); axes[0,i].set_xticks([])
    axes[1,i].imshow(mel,aspect='auto',origin='lower'); axes[1,i].set_title(rows10[i]['pred'],fontsize=8)
fig.tight_layout(); fig.savefig(FIG/'fs10_asr.png',dpi=120); plt.close()
(RES/'fs10.json').write_text(json.dumps({'stage':'FS10','method':'logmel CNN/BiGRU','acc':acc10,'history':hist10,'samples':rows10,'vs_prev':'video->audio'},indent=2))
PROGRESS['FS10']='ok'


In [ ]:
(RES/'summary_fs09_fs10.json').write_text(json.dumps({'progress':PROGRESS,'gates':GATES},indent=2))
(OUT/'SUCCESS').write_text('ok\n'); (OUT/'ACCEPTANCE.json').write_text(json.dumps({'ok':all(GATES.values()),'gates':GATES},indent=2))
print('FS09-10 PASS',GATES)
